# 12 — Dataset and DataLoader

In the previous notebook, we learned how optimizers update model parameters using gradients.

Now we will focus on another essential part of deep learning:

> **How data is organized, accessed, batched, shuffled, and fed into a model.**

For tiny examples, we can keep all tensors in memory and manually slice them.

For real deep-learning projects, this quickly becomes inconvenient.

PyTorch solves this with two important abstractions:

- `Dataset`
- `DataLoader`

## In this notebook, we will learn:

1. Why data pipelines matter
2. `Dataset`
3. `TensorDataset`
4. Custom `Dataset`
5. `__len__()`
6. `__getitem__()`
7. `DataLoader`
8. Batch size
9. Shuffling
10. Iterating through batches
11. Feature and target shapes
12. Train / validation / test datasets
13. `random_split()`
14. Mini-batch training
15. `num_workers`
16. Reproducible loading
17. `drop_last`
18. Common data-pipeline mistakes
19. Debugging data pipelines
20. Practice exercises

## Main Goal

By the end of this notebook, you should understand this pipeline:

$$
\boxed{
Dataset
\rightarrow
DataLoader
\rightarrow
Mini\text{-}Batches
\rightarrow
Model
}
$$

The most important habit is:

> **Always inspect the shape and dtype of one batch before starting a long training run.**


In [ ]:
import torch
from torch import nn
from torch.utils.data import (
    Dataset,
    TensorDataset,
    DataLoader,
    random_split
)

print("PyTorch version:", torch.__version__)


# 1. Why Data Pipelines Matter

A machine-learning model rarely receives the entire dataset as one giant tensor.

Real datasets may contain:

- Thousands of samples
- Millions of samples
- Images stored on disk
- Audio files
- Text
- Labels
- Multiple input features
- Data transformations

A good data pipeline should make it easy to:

- Access one sample
- Create batches
- Shuffle training data
- Separate training and evaluation data
- Load data efficiently
- Keep data handling separate from model code


# 2. Dataset vs DataLoader

These two objects have different responsibilities.

$$
\begin{array}{|c|c|}
\hline
\textbf{Dataset} & \textbf{DataLoader} \\
\hline
\text{Defines how to access samples} & \text{Creates batches} \\
\hline
\text{Knows dataset length} & \text{Can shuffle samples} \\
\hline
\text{Returns one sample at a time} & \text{Iterates over many batches} \\
\hline
\text{Stores or references data} & \text{Handles loading strategy} \\
\hline
\end{array}
$$

A useful mental model is:

> `Dataset` answers: **What is sample number i?**

> `DataLoader` answers: **How should I iterate through many samples?**


# 3. A Small Tensor Dataset

Let's begin with a very small supervised-learning dataset.

Suppose each sample has two input features.

$$
X =
\begin{array}{|c|c|}
\hline
1 & 2 \\
\hline
3 & 4 \\
\hline
5 & 6 \\
\hline
7 & 8 \\
\hline
9 & 10 \\
\hline
\end{array}
$$

and targets:

$$
y =
\begin{array}{|c|}
\hline
0 \\
\hline
1 \\
\hline
0 \\
\hline
1 \\
\hline
0 \\
\hline
\end{array}
$$

So:

$$
X.shape=(5,\ 2)
$$

and:

$$
y.shape=(5)
$$


In [ ]:
X = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
    [7.0, 8.0],
    [9.0, 10.0]
])

y = torch.tensor([
    0,
    1,
    0,
    1,
    0
])

print("X shape:", X.shape)
print("y shape:", y.shape)


# 4. What Is a `Dataset`?

A PyTorch `Dataset` represents a collection of samples.

A dataset should usually support two operations:

1. Tell us how many samples exist
2. Return a sample when given an index

These correspond to:

- `__len__()`
- `__getitem__()`


# 5. `TensorDataset`

If your data already exists as tensors, PyTorch provides:

`TensorDataset`

It is a convenient way to combine multiple tensors into one dataset.


In [ ]:
dataset = TensorDataset(
    X,
    y
)

print(dataset)


# 6. Length of a `TensorDataset`

Python's:

`len(dataset)`

calls the dataset's `__len__()` method internally.


In [ ]:
print("Number of samples:", len(dataset))


Our dataset contains:

$$
\boxed{5}
$$

samples.


# 7. Accessing One Sample

Indexing a dataset:

```python
dataset[0]
```

calls its `__getitem__()` method.

For a `TensorDataset`, each returned item is a tuple containing corresponding entries from the tensors.


In [ ]:
sample = dataset[0]

print(sample)


Let's separate the returned feature tensor and target.


In [ ]:
features, target = dataset[0]

print("Features:", features)
print("Target:", target)

print("Feature shape:", features.shape)
print("Target shape:", target.shape)


Notice the important difference:

The complete feature tensor has shape:

$$
(5,\ 2)
$$

but one sample has shape:

$$
(2)
$$

The dataset removes the sample dimension when one item is indexed.


# 8. Inspecting Several Samples


In [ ]:
for index in range(len(dataset)):
    features, target = dataset[index]

    print(
        f"Index {index} | "
        f"features = {features.tolist()} | "
        f"target = {target.item()}"
    )


# 9. Writing a Custom `Dataset`

For real projects, data may not already be stored in convenient tensors.

We can create our own dataset by inheriting from:

`torch.utils.data.Dataset`

A custom dataset commonly defines:

```python
class MyDataset(Dataset):
    def __init__(self, ...):
        ...

    def __len__(self):
        ...

    def __getitem__(self, index):
        ...
```


# 10. Custom Dataset Example

Let's rebuild the previous dataset ourselves.


In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        x = self.features[index]
        y = self.targets[index]

        return x, y

custom_dataset = SimpleDataset(
    X,
    y
)

print(custom_dataset)


# 11. Understanding `__init__()`

The dataset's `__init__()` method stores information needed later.

In our example:

```python
self.features = features
self.targets = targets
```

For an image dataset, `__init__()` might instead store:

- File paths
- Labels
- Transform functions
- Metadata

It does not necessarily need to load every image into memory immediately.


# 12. Understanding `__len__()`

`__len__()` should return the number of samples.


In [ ]:
print(
    "Dataset length:",
    len(custom_dataset)
)


For paired features and targets, their first dimensions should usually agree.


In [ ]:
print(
    "Feature samples:",
    X.shape[0]
)

print(
    "Target samples:",
    y.shape[0]
)


# 13. Understanding `__getitem__()`

`__getitem__(index)` defines what should be returned for one sample.

For supervised learning, a common return value is:

```python
features, target
```


In [ ]:
features, target = custom_dataset[2]

print("Features:", features)
print("Target:", target)


# 14. Why Custom Datasets Are Powerful

A custom dataset can perform work only when a sample is requested.

For example, `__getitem__()` can:

1. Read an image file
2. Decode the image
3. Apply preprocessing
4. Apply augmentation
5. Convert the label
6. Return tensors

This allows large datasets to be used without loading everything into memory at once.


# 15. A Dataset With On-the-Fly Transformation

Let's create a simple transformation that doubles the features.


In [ ]:
class TransformDataset(Dataset):
    def __init__(
        self,
        features,
        targets,
        transform=None
    ):
        self.features = features
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        x = self.features[index]
        y = self.targets[index]

        if self.transform is not None:
            x = self.transform(x)

        return x, y


In [ ]:
def double_features(x):
    return x * 2

transformed_dataset = TransformDataset(
    X,
    y,
    transform=double_features
)

original_x, _ = custom_dataset[0]
transformed_x, _ = transformed_dataset[0]

print("Original:", original_x)
print("Transformed:", transformed_x)


# 16. What Is a `DataLoader`?

A `DataLoader` wraps a dataset and provides an iterable over batches.

Basic syntax:

```python
loader = DataLoader(
    dataset,
    batch_size=...,
    shuffle=...
)
```

The `DataLoader` does not define what a sample means.

The `Dataset` does that.

The `DataLoader` controls how samples are grouped and iterated.


# 17. Creating Our First DataLoader

Let's use batch size:

$$
2
$$


In [ ]:
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False
)

print(loader)


# 18. Iterating Through Batches


In [ ]:
for batch_index, (
    batch_features,
    batch_targets
) in enumerate(loader):

    print(
        f"Batch {batch_index}"
    )

    print("Features:")
    print(batch_features)

    print("Targets:")
    print(batch_targets)

    print()


Our dataset has:

$$
5
$$

samples and batch size:

$$
2
$$

So the batches contain:

$$
2,\ 2,\ 1
$$

samples.

The last batch is smaller because 5 is not divisible by 2.


# 19. Batch Shape Reasoning

The original feature tensor has shape:

$$
(5,\ 2)
$$

Each individual feature sample has shape:

$$
(2)
$$

With:

$$
batch\_size=2
$$

a normal full batch has shape:

$$
\boxed{(2,\ 2)}
$$

The first dimension is the batch dimension.


In [ ]:
batch_features, batch_targets = next(
    iter(loader)
)

print(
    "Batch feature shape:",
    batch_features.shape
)

print(
    "Batch target shape:",
    batch_targets.shape
)


# 20. Understanding Batch Size

`batch_size` controls how many samples are grouped together.

Suppose:

$$
N=100
$$

samples.

If:

$$
batch\_size=10
$$

then there are approximately:

$$
\frac{100}{10}=10
$$

batches per epoch.

If the dataset size is not divisible by batch size, the last batch may be smaller.


# 21. Number of Batches

`len(loader)` gives the number of batches, not the number of samples.


In [ ]:
print(
    "Samples:",
    len(dataset)
)

print(
    "Batches:",
    len(loader)
)


For:

$$
5
$$

samples with batch size:

$$
2
$$

we get:

$$
\boxed{3}
$$

batches.


# 22. `drop_last`

Sometimes we want every batch to have exactly the same size.

Use:

```python
drop_last=True
```

Then the final incomplete batch is discarded.


In [ ]:
loader_drop_last = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    drop_last=True
)

for batch_index, (
    batch_features,
    batch_targets
) in enumerate(loader_drop_last):

    print(
        f"Batch {batch_index} shape:",
        batch_features.shape
    )


Now only:

$$
2
$$

full batches are returned.

One sample is dropped for that epoch.

Use `drop_last=True` only when you intentionally want this behavior.


# 23. Why Shuffle Training Data?

If training examples always appear in exactly the same order, batches may contain unwanted structure.

Shuffling helps randomize sample order between epochs.

Typical training loader:

```python
DataLoader(
    train_dataset,
    batch_size=...,
    shuffle=True
)
```


# 24. Shuffling Example

We will use sample numbers so the order is easy to see.


In [ ]:
numbers = torch.arange(
    10
)

number_dataset = TensorDataset(
    numbers
)

shuffled_loader = DataLoader(
    number_dataset,
    batch_size=5,
    shuffle=True
)

for batch in shuffled_loader:
    print(batch[0])


Run the previous cell more than once.

The order can change because:

`shuffle=True`

randomizes the sample order.


# 25. Should Validation and Test Data Be Shuffled?

Usually:

- Training → `shuffle=True`
- Validation → `shuffle=False`
- Test → `shuffle=False`

Why?

Training benefits from randomized mini-batches.

For validation and testing, deterministic order is usually easier for:

- Debugging
- Matching predictions to samples
- Reproducibility

Shuffling evaluation data normally does not improve the model.


# 26. Iterators and `next()`

A `DataLoader` is iterable.

We can create an iterator:

```python
iterator = iter(loader)
```

and request the next batch:

```python
batch = next(iterator)
```


In [ ]:
iterator = iter(loader)

first_batch = next(iterator)

print(first_batch)


# 27. Unpacking a Batch

For supervised learning, a batch commonly contains:

```python
inputs, targets
```


In [ ]:
inputs, targets = first_batch

print("Inputs:")
print(inputs)

print()

print("Targets:")
print(targets)


# 28. Feature and Target Shapes

Before training, always inspect:

```python
inputs.shape
targets.shape
inputs.dtype
targets.dtype
```

This can catch many bugs before the model is involved.


In [ ]:
inputs, targets = next(
    iter(loader)
)

print(
    "Input shape:",
    inputs.shape
)

print(
    "Target shape:",
    targets.shape
)

print(
    "Input dtype:",
    inputs.dtype
)

print(
    "Target dtype:",
    targets.dtype
)


# 29. Shape Depends on the Task

Suppose we have images.

A batch might have shape:

$$
\boxed{
(batch,\ channels,\ height,\ width)
}
$$

Example:

$$
\boxed{
(16,\ 1,\ 256,\ 256)
}
$$

For grayscale ultrasound images:

$$
\begin{array}{|c|c|}
\hline
16 & \text{Batch size} \\
\hline
1 & \text{Channel} \\
\hline
256 & \text{Height} \\
\hline
256 & \text{Width} \\
\hline
\end{array}
$$

Targets for binary classification might have shape:

$$
(16)
$$

or:

$$
(16,\ 1)
$$

depending on the model/loss convention.


# 30. Simulated Image Dataset

Let's create fake grayscale images.

Each image:

$$
(1,\ 64,\ 64)
$$

Number of images:

$$
100
$$


In [ ]:
images = torch.randn(
    100,
    1,
    64,
    64
)

labels = torch.randint(
    0,
    2,
    (100,)
)

image_dataset = TensorDataset(
    images,
    labels
)

image_loader = DataLoader(
    image_dataset,
    batch_size=16,
    shuffle=True
)


In [ ]:
image_batch, label_batch = next(
    iter(image_loader)
)

print(
    "Image batch:",
    image_batch.shape
)

print(
    "Label batch:",
    label_batch.shape
)


The image batch has shape:

$$
\boxed{(16,\ 1,\ 64,\ 64)}
$$

This is the standard PyTorch image ordering:

$$
\boxed{N,\ C,\ H,\ W}
$$

where:

- $N$ = batch
- $C$ = channels
- $H$ = height
- $W$ = width


# 31. Train / Validation / Test Sets

A common machine-learning workflow separates data into:

- Training set
- Validation set
- Test set

$$
\begin{array}{|c|c|}
\hline
\textbf{Split} & \textbf{Purpose} \\
\hline
Training & \text{Learn model parameters} \\
\hline
Validation & \text{Tune choices and monitor generalization} \\
\hline
Test & \text{Final unbiased evaluation} \\
\hline
\end{array}
$$

The exact percentages depend on the problem and dataset size.


# 32. Why We Need Separate Splits

If we evaluate only on training data, we do not know whether the model generalizes to unseen examples.

Validation data helps us make development decisions.

Test data should be protected from repeated tuning.

The key principle is:

> **Do not train on validation or test samples.**


# 33. Splitting a Dataset With `random_split()`

Let's create a dataset containing 100 samples.

We will split it into:

- 70 training
- 15 validation
- 15 test


In [ ]:
features = torch.randn(
    100,
    5
)

targets = torch.randint(
    0,
    3,
    (100,)
)

full_dataset = TensorDataset(
    features,
    targets
)

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [70, 15, 15],
    generator=torch.Generator().manual_seed(42)
)

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Test:",
    len(test_dataset)
)


# 34. Why Use a Seed When Splitting?

Without a fixed random seed, the exact samples assigned to each split can change between runs.

For reproducible experiments, we often fix the split using:

```python
generator=torch.Generator().manual_seed(...)
```

Then the same split can be recreated.


# 35. Separate DataLoaders for Each Split

A common pattern is:

- Train loader → shuffle
- Validation loader → no shuffle
- Test loader → no shuffle


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)


# 36. Inspecting One Training Batch


In [ ]:
train_features, train_targets = next(
    iter(train_loader)
)

print(
    "Features:",
    train_features.shape
)

print(
    "Targets:",
    train_targets.shape
)

print(
    "Target dtype:",
    train_targets.dtype
)


# 37. Mini-Batch Training

Now let's connect a `DataLoader` to a training loop.

Suppose we have:

$$
5
$$

input features and:

$$
3
$$

classes.

Our model can be:

$$
5
\rightarrow
16
\rightarrow
3
$$

For multi-class classification:

- Output = 3 logits
- Target = class index
- Loss = `CrossEntropyLoss`


In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(5, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

print(model)


# 38. One Mini-Batch Training Step


In [ ]:
model.train()

batch_features, batch_targets = next(
    iter(train_loader)
)

optimizer.zero_grad()

logits = model(
    batch_features
)

loss = criterion(
    logits,
    batch_targets
)

loss.backward()

optimizer.step()

print(
    "Batch features:",
    batch_features.shape
)

print(
    "Batch targets:",
    batch_targets.shape
)

print(
    "Logits:",
    logits.shape
)

print(
    "Loss:",
    loss.item()
)


# 39. One Complete Training Epoch

An **epoch** means one pass through the entire training dataset.

With a `DataLoader`, that means iterating through all training batches once.


In [ ]:
model.train()

running_loss = 0.0

for batch_features, batch_targets in train_loader:
    optimizer.zero_grad()

    logits = model(
        batch_features
    )

    loss = criterion(
        logits,
        batch_targets
    )

    loss.backward()

    optimizer.step()

    running_loss += (
        loss.item()
        * batch_features.size(0)
    )

epoch_loss = (
    running_loss
    / len(train_loader.dataset)
)

print(
    "Training loss:",
    epoch_loss
)


# 40. Why Multiply Batch Loss by Batch Size?

`CrossEntropyLoss` uses mean reduction by default.

So each batch loss is the average loss within that batch.

If the final batch is smaller, simply averaging batch averages gives every batch equal importance.

A more accurate dataset-level average is:

$$
\frac{
\sum
(batch\_mean\_loss \times batch\_size)
}{
number\_of\_samples
}
$$

This correctly weights each sample.


# 41. Validation Loop

Validation does not update parameters.

Use:

- `model.eval()`
- `torch.no_grad()`


In [ ]:
model.eval()

validation_loss_sum = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_targets in val_loader:
        logits = model(
            batch_features
        )

        loss = criterion(
            logits,
            batch_targets
        )

        validation_loss_sum += (
            loss.item()
            * batch_features.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions
            == batch_targets
        ).sum().item()

        total += (
            batch_targets.size(0)
        )

validation_loss = (
    validation_loss_sum
    / len(val_loader.dataset)
)

validation_accuracy = (
    correct / total
)

print(
    "Validation loss:",
    validation_loss
)

print(
    "Validation accuracy:",
    validation_accuracy
)


# 42. A Clean Multi-Epoch Training Loop

Let's combine training and validation.

The random synthetic labels in this example do not contain a meaningful learnable pattern, so the purpose here is to understand the **data pipeline and loop structure**, not to achieve high accuracy.


In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(5, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

epochs = 5

for epoch in range(epochs):
    # ---------------------
    # Training
    # ---------------------
    model.train()

    train_loss_sum = 0.0

    for batch_features, batch_targets in train_loader:
        optimizer.zero_grad()

        logits = model(
            batch_features
        )

        loss = criterion(
            logits,
            batch_targets
        )

        loss.backward()

        optimizer.step()

        train_loss_sum += (
            loss.item()
            * batch_features.size(0)
        )

    train_loss = (
        train_loss_sum
        / len(train_loader.dataset)
    )

    # ---------------------
    # Validation
    # ---------------------
    model.eval()

    val_loss_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_features, batch_targets in val_loader:
            logits = model(
                batch_features
            )

            loss = criterion(
                logits,
                batch_targets
            )

            val_loss_sum += (
                loss.item()
                * batch_features.size(0)
            )

            predictions = logits.argmax(
                dim=1
            )

            correct += (
                predictions
                == batch_targets
            ).sum().item()

            total += (
                batch_targets.size(0)
            )

    val_loss = (
        val_loss_sum
        / len(val_loader.dataset)
    )

    val_accuracy = (
        correct / total
    )

    print(
        f"Epoch {epoch + 1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.3f}"
    )


# 43. `num_workers`

`DataLoader` can use worker processes to load data.

Example:

```python
DataLoader(
    dataset,
    batch_size=32,
    num_workers=2
)
```

`num_workers` controls how many worker processes are used for loading.

A simple interpretation:

$$
\begin{array}{|c|c|}
\hline
num\_workers=0 & \text{Load in main process} \\
\hline
num\_workers>0 & \text{Use worker processes} \\
\hline
\end{array}
$$


# 44. Choosing `num_workers`

There is no universally best value.

It depends on:

- Operating system
- CPU
- Storage speed
- Dataset processing
- Batch size
- Runtime environment

For beginner notebooks and debugging:

```python
num_workers=0
```

is a safe starting point.

For larger real datasets, benchmark higher values.


In [ ]:
safe_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

print(
    "num_workers:",
    safe_loader.num_workers
)


# 45. Why More Workers Are Not Always Better

Increasing `num_workers` can improve throughput when data loading is a bottleneck.

But too many workers can:

- Increase memory usage
- Add process overhead
- Cause environment-specific issues
- Make debugging harder

So:

> **Measure performance instead of assuming a larger number is better.**


# 46. Reproducible Shuffling

If:

`shuffle=True`

we may still want reproducible batch ordering.

We can provide a seeded `torch.Generator`.


In [ ]:
generator = torch.Generator()
generator.manual_seed(123)

reproducible_loader = DataLoader(
    number_dataset,
    batch_size=5,
    shuffle=True,
    generator=generator
)

for batch in reproducible_loader:
    print(batch[0])


# 47. Recreating the Same Shuffle

To reproduce the same first epoch ordering, create a new generator with the same seed.


In [ ]:
generator_a = torch.Generator().manual_seed(
    123
)

generator_b = torch.Generator().manual_seed(
    123
)

loader_a = DataLoader(
    number_dataset,
    batch_size=5,
    shuffle=True,
    generator=generator_a
)

loader_b = DataLoader(
    number_dataset,
    batch_size=5,
    shuffle=True,
    generator=generator_b
)

order_a = torch.cat([
    batch[0]
    for batch in loader_a
])

order_b = torch.cat([
    batch[0]
    for batch in loader_b
])

print("Order A:", order_a)
print("Order B:", order_b)

print(
    "Same order:",
    torch.equal(
        order_a,
        order_b
    )
)


# 48. Reproducibility With Multiple Workers

When `num_workers > 0`, each worker may also use random-number generators.

For advanced pipelines with random NumPy or Python operations inside workers, reproducibility may require explicitly seeding worker processes.

The important beginner idea is:

> **Reproducibility involves more than only calling `torch.manual_seed()`.**

It can involve:

- Dataset split seed
- DataLoader shuffle generator
- Worker seeds
- Model initialization seed
- CUDA determinism settings

We will study reproducibility more deeply later.


# 49. `pin_memory`

For GPU training, you may see:

```python
pin_memory=True
```

in a `DataLoader`.

Pinned CPU memory can sometimes make CPU-to-GPU transfers faster.

It is most relevant when:

- Training on CUDA
- Data is loaded on CPU
- Transfer speed matters

For simple CPU examples, it is usually unnecessary.


In [ ]:
pin_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)

print(
    "pin_memory:",
    pin_loader.pin_memory
)


# 50. Moving Batches to a Device

The `DataLoader` usually returns CPU tensors.

During GPU training, batches are moved to the model's device.

Typical pattern:

```python
inputs = inputs.to(device)
targets = targets.to(device)
```


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

device_model = nn.Sequential(
    nn.Linear(5, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
).to(device)

batch_features, batch_targets = next(
    iter(train_loader)
)

batch_features = batch_features.to(
    device
)

batch_targets = batch_targets.to(
    device
)

print(
    "Model device:",
    next(
        device_model.parameters()
    ).device
)

print(
    "Feature device:",
    batch_features.device
)

print(
    "Target device:",
    batch_targets.device
)


# 51. Custom Dataset for File Paths — Concept

A real image dataset often stores file paths instead of all image tensors.

Conceptually:

```python
class ImageDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        label = self.labels[index]

        image = load_image(path)

        if self.transform is not None:
            image = self.transform(image)

        return image, label
```

The file is loaded only when the corresponding sample is requested.


# 52. Why Lazy Loading Matters

Suppose you have:

$$
50,000
$$

ultrasound images.

Loading all of them into RAM at once may be unnecessary or impossible.

A file-based custom dataset can:

1. Keep only file paths in memory
2. Load one sample when requested
3. Apply preprocessing
4. Return a tensor
5. Let `DataLoader` build batches

This is the standard pattern for many medical-imaging projects.


# 53. Dataset Transformations

A transformation can perform operations such as:

- Resize
- Normalize
- Crop
- Augment
- Convert dtype
- Rearrange dimensions

A good design often keeps transformations inside or associated with the dataset pipeline rather than scattering preprocessing throughout the training loop.


# 54. Training vs Validation Transformations

Training data may use random augmentation.

Validation and test data should generally use deterministic preprocessing.

For example:

$$
\begin{array}{|c|c|}
\hline
\textbf{Training} & \textbf{Validation/Test} \\
\hline
Random\ crop & Resize/center\ crop \\
\hline
Random\ flip & No\ random\ flip \\
\hline
Augmentation & Deterministic\ preprocessing \\
\hline
Normalization & Same\ compatible\ normalization \\
\hline
\end{array}
$$

Validation should measure model performance, not randomness from training augmentation.


# 55. Avoiding Data Leakage

One of the most dangerous data-pipeline mistakes is:

> **Data leakage**

Data leakage occurs when information from validation or test data improperly influences training.

Examples:

- Same patient appearing in both train and test
- Normalization statistics computed from the full dataset when they should come only from training data
- Augmented copies of the same original sample split across train and test
- Test data used repeatedly for model selection

For medical imaging, patient-level splitting is often especially important.


# 56. Sample-Level vs Group-Level Splitting

Suppose one patient has multiple ultrasound images.

If individual images are randomly split, images from the same patient could appear in:

- Training
- Validation
- Test

That can produce overly optimistic results.

When samples are grouped by patient, subject, study, or acquisition session, the split may need to happen at that group level.

The correct split unit depends on the scientific problem.


# 57. Common Mistake — Features and Targets Have Different Lengths

This is invalid:

$$
X.shape=(100,\ 5)
$$

and:

$$
y.shape=(90)
$$

because ten samples have no corresponding target.

Always check:

```python
len(features) == len(targets)
```


In [ ]:
features = torch.randn(
    100,
    5
)

targets = torch.randn(
    100
)

assert (
    len(features)
    == len(targets)
)

print("Lengths match.")


# 58. Common Mistake — Wrong Target Dtype

The correct dtype depends on the loss.

For standard multi-class classification with:

`CrossEntropyLoss`

targets are commonly:

`torch.long`

For binary classification with:

`BCEWithLogitsLoss`

targets are commonly floating point.

The data pipeline should return targets in the form expected by the loss.


# 59. Common Mistake — Shuffling Validation or Test Data Without a Reason

Shuffling evaluation data usually gives no benefit and can make it harder to:

- Reproduce ordering
- Match predictions to sample IDs
- Debug specific errors

A common choice is:

```python
shuffle=False
```

for validation and test loaders.


# 60. Common Mistake — Applying Random Augmentation to Validation

Random training augmentation helps create diverse training examples.

But validation and test sets should usually represent a stable evaluation pipeline.

If random transformations change every evaluation pass, metrics can become unnecessarily noisy.


# 61. Common Mistake — Assuming Every Batch Has the Requested Batch Size

If:

$$
N
$$

is not divisible by:

$$
batch\_size
$$

the final batch may be smaller.

Never blindly assume:

```python
inputs.shape[0] == batch_size
```

unless you intentionally use:

`drop_last=True`


# 62. Common Mistake — Forgetting the Batch Dimension

A model may expect:

$$
(batch,\ features)
$$

but one raw dataset sample may have shape:

$$
(features)
$$

The `DataLoader` automatically stacks individual samples into batches when possible.

This is one reason the shape after `DataLoader` can differ from `dataset[index]`.


# 63. Common Mistake — Returning Inconsistent Shapes

Suppose one image sample has shape:

$$
(1,\ 256,\ 256)
$$

and another has:

$$
(1,\ 300,\ 240)
$$

Default batching cannot simply stack them into one rectangular tensor.

You may need:

- Resizing
- Padding
- A custom `collate_fn`
- Task-specific batching

Consistent sample shapes are important for standard batching.


# 64. `collate_fn` — Brief Introduction

A `DataLoader` uses a collation step to combine individual samples into a batch.

The default behavior works well for tensors with compatible shapes.

For unusual data structures or variable-size samples, you can provide:

`collate_fn=...`

We will not go deeply into custom collation yet, but remember:

> **Dataset returns samples. Collation combines samples into batches.**


# 65. Data-Pipeline Debugging Checklist

Before training, inspect:

1. `len(dataset)`
2. `dataset[0]`
3. Feature shape
4. Target shape
5. Feature dtype
6. Target dtype
7. `len(loader)`
8. One complete batch
9. Batch dimension
10. Whether shuffling is appropriate
11. Whether train/val/test splits overlap
12. Whether transformations differ appropriately
13. Device movement
14. Last-batch behavior


In [ ]:
sample_features, sample_target = (
    train_dataset[0]
)

print(
    "Single feature shape:",
    sample_features.shape
)

print(
    "Single target shape:",
    sample_target.shape
)

batch_features, batch_targets = next(
    iter(train_loader)
)

print(
    "Batch feature shape:",
    batch_features.shape
)

print(
    "Batch target shape:",
    batch_targets.shape
)

print(
    "Batch feature dtype:",
    batch_features.dtype
)

print(
    "Batch target dtype:",
    batch_targets.dtype
)


# 66. A Reusable Loader-Building Function

We can reduce repeated code with a helper function.


In [ ]:
def make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32,
    num_workers=0,
    seed=42
):
    generator = (
        torch.Generator()
        .manual_seed(seed)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )


In [ ]:
train_loader_2, val_loader_2, test_loader_2 = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=8
)

print(
    len(train_loader_2),
    len(val_loader_2),
    len(test_loader_2)
)


# 67. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

Create features:

$$
X.shape=(20,\ 4)
$$

and binary targets:

$$
y.shape=(20)
$$

Wrap them in a `TensorDataset`.

## Exercise 2

Check:

- Dataset length
- First sample
- First feature shape
- First target shape

## Exercise 3

Create a `DataLoader` with:

$$
batch\_size=5
$$

and:

`shuffle=True`

## Exercise 4

How many batches should 20 samples create with batch size 5?

## Exercise 5

Create 22 samples with batch size 5.

How many samples should the final batch contain?

## Exercise 6

Repeat Exercise 5 with:

`drop_last=True`

How many batches remain?

## Exercise 7

Write a custom dataset with:

- `__init__()`
- `__len__()`
- `__getitem__()`

## Exercise 8

Split 100 samples into:

- 70 train
- 15 validation
- 15 test

using a fixed seed.

## Exercise 9

Create separate train, validation, and test loaders with appropriate shuffle settings.

## Exercise 10

Write one mini-batch training epoch using a `DataLoader`.


# 68. Shape Reasoning Challenges

Answer before running code.

## Challenge 1

A dataset contains:

$$
X.shape=(1000,\ 10)
$$

with batch size:

$$
32
$$

What is the normal full-batch feature shape?

## Challenge 2

For the same dataset, what can the final batch size be if:

`drop_last=False`?

## Challenge 3

An image dataset stores samples with shape:

$$
(1,\ 256,\ 256)
$$

What is the batch shape for batch size 16?

## Challenge 4

A 10-class classifier uses:

`CrossEntropyLoss`

For batch size 64, what are the usual:

- Logit shape
- Target shape
- Target dtype

## Challenge 5

Why can a single dataset sample have shape:

$$
(5)
$$

while a `DataLoader` batch has shape:

$$
(32,\ 5)
$$

## Challenge 6

Why should data splitting sometimes happen at the patient level rather than image level?


# 69. Exercise Solutions


In [ ]:
# Exercise 1
X_ex = torch.randn(
    20,
    4
)

y_ex = torch.randint(
    0,
    2,
    (20,)
)

dataset_ex = TensorDataset(
    X_ex,
    y_ex
)

print(
    "Exercise 1:",
    len(dataset_ex)
)

# Exercise 2
x0, y0 = dataset_ex[0]

print(
    "Exercise 2 feature:",
    x0.shape
)

print(
    "Exercise 2 target:",
    y0.shape
)

# Exercise 3
loader_ex = DataLoader(
    dataset_ex,
    batch_size=5,
    shuffle=True
)

print(
    "Exercise 3 batches:",
    len(loader_ex)
)

# Exercise 4
print(
    "Exercise 4:",
    len(loader_ex)
)

# Exercise 5
dataset_22 = TensorDataset(
    torch.randn(22, 4),
    torch.randint(
        0,
        2,
        (22,)
    )
)

loader_22 = DataLoader(
    dataset_22,
    batch_size=5,
    shuffle=False
)

last_batch = None

for batch in loader_22:
    last_batch = batch

print(
    "Exercise 5 last batch size:",
    last_batch[0].shape[0]
)

# Exercise 6
loader_drop = DataLoader(
    dataset_22,
    batch_size=5,
    drop_last=True
)

print(
    "Exercise 6 batches:",
    len(loader_drop)
)

# Exercise 7
class ExerciseDataset(Dataset):
    def __init__(
        self,
        features,
        targets
    ):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(
            self.features
        )

    def __getitem__(
        self,
        index
    ):
        return (
            self.features[index],
            self.targets[index]
        )

exercise_custom = ExerciseDataset(
    X_ex,
    y_ex
)

print(
    "Exercise 7 length:",
    len(exercise_custom)
)

# Exercise 8
dataset_100 = TensorDataset(
    torch.randn(100, 4),
    torch.randint(
        0,
        2,
        (100,)
    )
)

train_ex, val_ex, test_ex = random_split(
    dataset_100,
    [70, 15, 15],
    generator=(
        torch.Generator()
        .manual_seed(42)
    )
)

print(
    "Exercise 8:",
    len(train_ex),
    len(val_ex),
    len(test_ex)
)

# Exercise 9
train_loader_ex = DataLoader(
    train_ex,
    batch_size=16,
    shuffle=True
)

val_loader_ex = DataLoader(
    val_ex,
    batch_size=16,
    shuffle=False
)

test_loader_ex = DataLoader(
    test_ex,
    batch_size=16,
    shuffle=False
)

print(
    "Exercise 9 complete."
)


# 70. Key Takeaways

In this notebook, we learned:

- Why data pipelines matter
- `Dataset`
- `TensorDataset`
- Custom datasets
- `__len__()`
- `__getitem__()`
- Transformations
- `DataLoader`
- Batch size
- Number of batches
- `drop_last`
- Shuffling
- Iterators
- Feature and target shapes
- Image batch shapes
- Train / validation / test splits
- `random_split()`
- Mini-batch training
- Validation loops
- `num_workers`
- Reproducible shuffling
- `pin_memory`
- Device movement
- Lazy file loading
- Data leakage
- Group-level splitting
- Common data-pipeline mistakes
- Data-pipeline debugging

The central pipeline is:

$$
\boxed{
Dataset
\rightarrow
DataLoader
\rightarrow
Batch
\rightarrow
Model
\rightarrow
Loss
}
$$

Always inspect one sample and one batch before training.


# 71. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is the responsibility of a `Dataset`?
2. What is the responsibility of a `DataLoader`?
3. What does `__len__()` do?
4. What does `__getitem__()` do?
5. What is `TensorDataset`?
6. Why would you create a custom dataset?
7. What does `batch_size` control?
8. What does `shuffle=True` do?
9. Why is training data usually shuffled?
10. Why are validation and test loaders usually not shuffled?
11. What does `len(loader)` return?
12. What happens to the last batch if the dataset size is not divisible by batch size?
13. What does `drop_last=True` do?
14. What is an epoch?
15. Why do we use train / validation / test splits?
16. What is data leakage?
17. Why might patient-level splitting matter in medical imaging?
18. What does `num_workers` control?
19. How can a seeded generator help reproducibility?
20. Why should you inspect one batch before training?


# Next Notebook

# 13 — Training and Validation Loops

In the next notebook, we will study:

- Anatomy of a training loop
- Training mode
- Validation mode
- Forward pass
- Loss accumulation
- Backward pass
- Optimizer step
- Batch metrics
- Epoch metrics
- `model.train()`
- `model.eval()`
- `torch.no_grad()`
- Tracking train vs validation loss
- Accuracy calculation
- Best-model checkpointing
- Early-stopping intuition
- Clean reusable training functions
